In [5]:
import pandas as pd # Librería de lectura de datos

import matplotlib.pyplot as plt # Librería de visualización de datos

from sklearn.model_selection import train_test_split,GridSearchCV # Función para dividir los datos en entrenamiento y prueba
from sklearn.preprocessing import MinMaxScaler,OneHotEncoder,LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier,plot_tree, export_text

from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, roc_curve # Funciones para evaluar el rendimiento del modelo

from sklearn.feature_selection import (VarianceThreshold, SelectKBest,
                                      f_classif, RFE, SelectFromModel)
from sklearn import svm
from sklearn.metrics import confusion_matrix as cm_func
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import confusion_matrix as cm_func2
from xgboost import XGBClassifier
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from sklearn.metrics import confusion_matrix as cm_func3

from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from sklearn.preprocessing import StandardScaler
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

df = pd.read_parquet('../clean_data/mushrooms_limpio.parquet')

df.info()
# Resumen del análisis
- **Forma:** (8124, 21)
- **Todas las columnas son categóricas** (tipo `str`).
- **Nulos:** 0 en todas las columnas.
- **Balance de clases:** `e` 51.8%, `p` 48.2% (casi balanceado).
- **Variabilidad:** varias columnas con baja cardinalidad (2–6 valores); algunas con más (hasta 12).

Recomendaciones rápidas:
- Para clustering con algoritmos basados en distancia (KMeans, KNN): convertir categóricas con `OneHotEncoder` (sparse), luego reducir dimensión (TruncatedSVD o UMAP) y aplicar `StandardScaler`.
- Si quieres evitar expansión dimensional: usar codificaciones por frecuencia / target encoding / hashing, o usar `KModes` / `k-prototypes` para datos categóricos.
- No escalar variables para métodos basados en árboles o `KModes`.
- Imputación: `SimpleImputer(strategy='most_frequent')` si aparecen nulos en otras fuentes.

He incluido a continuación un ejemplo de pipeline reproducible que puedes ejecutar (celda 3).

In [ ]:
# Cargar el dataset (intenta parquet, si falla usa CSV)
try:
    df = pd.read_parquet('../clean_data/mushrooms_limpio.parquet')
except Exception:
    df = pd.read_csv('../clean_data/mushrooms_limpio.csv')

cat_cols = df.select_dtypes(include=['object','category']).columns.tolist()

preproc = ColumnTransformer([
        ('imp_ohe', Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=True))
    ]), cat_cols),
], remainder='drop', sparse_threshold=0)

# Ajusta n_components según dimensionalidad (aquí 30 como ejemplo)
n_components = min(30, max(2, len(cat_cols)))
pipe = Pipeline([
    ('preproc', preproc),
    ('svd', TruncatedSVD(n_components=n_components, random_state=0)),
    ('scaler', StandardScaler()),
    ('km', KMeans(n_clusters=3, random_state=0))
])

pipe.fit(df)
labels = pipe.named_steps['km'].labels_
print('clusters assigned:', len(set(labels)))

# Guarda etiquetas
df['cluster'] = labels
df.to_csv('../clean_data/mushrooms_with_clusters.csv', index=False)
print('Wrote ../clean_data/mushrooms_with_clusters.csv')

clusters assigned: 3
Wrote ../clean_data/mushrooms_with_clusters.csv
